# Gold Layer — Business-Ready Aggregations

**Notebook:** `03_gold_aggregation`  
**Layer:** Gold (Curated / Business-Ready)  
**Lakehouse:** `RetailSalesLakehouse`  
**DP-700 Skill:** Ingest and Transform Data > Design Loading Patterns > Dimensional Model Preparation

## Purpose
Builds Gold layer tables optimized for business reporting and analytics:
- **`gold.dim_customer`** — Customer dimension with segments
- **`gold.dim_product`** — Product dimension with hierarchy
- **`gold.dim_date`** — Date dimension (static, pre-generated)
- **`gold.fact_orders`** — Order fact table
- **`gold.fact_order_items`** — Order line item fact table
- **`gold.agg_sales_daily`** — Pre-aggregated daily sales summary
- **`gold.agg_customer_lifetime`** — Customer lifetime value metrics

In [ ]:
load_mode = "full"
run_id = "manual"

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lit, current_timestamp, when, coalesce, trim,
    year, month, dayofmonth, dayofweek, quarter, weekofyear,
    date_format, sum as spark_sum, count, avg, min, max,
    countDistinct, round as spark_round, dense_rank, desc
)
from pyspark.sql.types import (
    IntegerType, DecimalType, StringType, DateType, BooleanType
)
from pyspark.sql.window import Window
from delta.tables import DeltaTable
import datetime

spark = SparkSession.builder.getOrCreate()

SILVER_SCHEMA = "silver"
GOLD_SCHEMA = "gold"

def write_gold(df, table_name, partition_by=None, zorder_cols=None):
    """Write Gold table with optional partitioning and Z-ordering."""
    writer = df.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
    if partition_by:
        writer = writer.partitionBy(*partition_by)
    writer.saveAsTable(f"{GOLD_SCHEMA}.{table_name}")
    if zorder_cols:
        spark.sql(f"OPTIMIZE {GOLD_SCHEMA}.{table_name} ZORDER BY ({', '.join(zorder_cols)})")
    count = spark.table(f"{GOLD_SCHEMA}.{table_name}").count()
    print(f"[SUCCESS] gold.{table_name}: {count:,} rows")

print("Gold layer build started")

## 1. dim_date — Date Dimension

In [ ]:
from pyspark.sql import Row

# Generate a date dimension from 2020-01-01 to 2026-12-31
start = datetime.date(2020, 1, 1)
end = datetime.date(2026, 12, 31)
delta = end - start
dates = [start + datetime.timedelta(days=i) for i in range(delta.days + 1)]

date_rows = [Row(date_key=int(d.strftime("%Y%m%d")), full_date=str(d)) for d in dates]
df_dates_raw = spark.createDataFrame(date_rows)

df_dim_date = (
    df_dates_raw
    .withColumn("full_date", col("full_date").cast(DateType()))
    .withColumn("year", year(col("full_date")))
    .withColumn("quarter", quarter(col("full_date")))
    .withColumn("month_number", month(col("full_date")))
    .withColumn("month_name", date_format(col("full_date"), "MMMM"))
    .withColumn("month_short", date_format(col("full_date"), "MMM"))
    .withColumn("week_of_year", weekofyear(col("full_date")))
    .withColumn("day_of_month", dayofmonth(col("full_date")))
    .withColumn("day_of_week", dayofweek(col("full_date")))
    .withColumn("day_name", date_format(col("full_date"), "EEEE"))
    .withColumn("is_weekend", dayofweek(col("full_date")).isin([1, 7]))
    .withColumn("year_month", date_format(col("full_date"), "yyyy-MM"))
    .withColumn("year_quarter", 
        col("year").cast(StringType()) + lit("-Q") + col("quarter").cast(StringType())
    )
)

write_gold(df_dim_date, "dim_date")

## 2. dim_customer — Customer Dimension

In [ ]:
df_customers = spark.table(f"{SILVER_SCHEMA}.customers")

df_dim_customer = (
    df_customers
    .filter(col("_dq_email_valid") == True)  # Only valid customers
    .select(
        "customer_id",
        "first_name",
        "last_name",
        "full_name",
        "email",
        "gender",
        "date_of_birth",
        "city",
        "state",
        "country",
        "postal_code",
        "segment",
        "registration_date",
        "is_active",
    )
    .withColumn("_gold_updated_at", current_timestamp())
)

write_gold(df_dim_customer, "dim_customer", zorder_cols=["customer_id"])

## 3. dim_product — Product Dimension

In [ ]:
df_products = spark.table(f"{SILVER_SCHEMA}.products")

df_dim_product = (
    df_products
    .select(
        "product_id",
        "product_name",
        "category",
        "subcategory",
        "brand",
        "unit_price",
        "cost_price",
        "margin_pct",
        "weight_kg",
        "is_active",
        "created_date",
    )
    .withColumn("price_band",
        when(col("unit_price") < 25, "Budget (<$25)")
        .when(col("unit_price") < 75, "Mid-Range ($25-$75)")
        .when(col("unit_price") < 150, "Premium ($75-$150)")
        .otherwise("Luxury (>$150)")
    )
    .withColumn("_gold_updated_at", current_timestamp())
)

write_gold(df_dim_product, "dim_product", zorder_cols=["product_id"])

## 4. fact_orders — Order Fact Table

In [ ]:
df_orders = spark.table(f"{SILVER_SCHEMA}.orders")
df_dim_date_ref = spark.table(f"{GOLD_SCHEMA}.dim_date").select("full_date", "date_key")

df_fact_orders = (
    df_orders
    .filter(col("status") != "Cancelled")  # Exclude cancelled orders
    # Join to date dimension for date surrogate keys
    .join(df_dim_date_ref.alias("order_dk"), col("order_date") == col("order_dk.full_date"), "left")
    .withColumnRenamed("date_key", "order_date_key")
    .drop("full_date")
    .join(df_dim_date_ref.alias("ship_dk"), col("ship_date") == col("ship_dk.full_date"), "left")
    .withColumnRenamed("date_key", "ship_date_key")
    .drop("full_date")
    .select(
        "order_id",
        "customer_id",
        "order_date",
        "order_date_key",
        "ship_date",
        "ship_date_key",
        "delivery_date",
        "status",
        "payment_method",
        "shipping_method",
        "channel",
        "coupon_code",
        "shipping_cost",
        "discount_amount",
        "subtotal",
        "tax_amount",
        "total_amount",
        "days_to_ship",
        "days_to_deliver",
    )
    .withColumn("has_discount", col("discount_amount") > 0)
    .withColumn("_gold_updated_at", current_timestamp())
)

write_gold(df_fact_orders, "fact_orders",
           partition_by=["order_date"],
           zorder_cols=["customer_id", "order_id"])

## 5. fact_order_items — Line Item Fact Table

In [ ]:
df_items = spark.table(f"{SILVER_SCHEMA}.order_items")
df_orders_ref = spark.table(f"{SILVER_SCHEMA}.orders").select(
    "order_id", "customer_id", "order_date", "status", "channel"
)

df_fact_items = (
    df_items
    .join(df_orders_ref, "order_id", "inner")
    .filter(col("status") != "Cancelled")
    .select(
        "order_item_id",
        "order_id",
        "product_id",
        "customer_id",
        "order_date",
        "channel",
        "quantity",
        "unit_price",
        "discount_pct",
        "line_total",
    )
    .withColumn("_gold_updated_at", current_timestamp())
)

write_gold(df_fact_items, "fact_order_items",
           partition_by=["order_date"],
           zorder_cols=["order_id", "product_id"])

## 6. agg_sales_daily — Pre-Aggregated Daily Sales

In [ ]:
df_daily_sales = (
    spark.table(f"{GOLD_SCHEMA}.fact_orders")
    .groupBy("order_date", "channel", "shipping_method", "payment_method")
    .agg(
        count("order_id").alias("order_count"),
        countDistinct("customer_id").alias("unique_customers"),
        spark_sum("total_amount").alias("total_revenue"),
        spark_sum("discount_amount").alias("total_discounts"),
        spark_sum("shipping_cost").alias("total_shipping"),
        avg("total_amount").alias("avg_order_value"),
        spark_sum(when(col("has_discount"), 1).otherwise(0)).alias("discounted_orders"),
    )
    .withColumn("total_revenue", spark_round("total_revenue", 2))
    .withColumn("avg_order_value", spark_round("avg_order_value", 2))
    .withColumn("_gold_updated_at", current_timestamp())
)

write_gold(df_daily_sales, "agg_sales_daily", partition_by=["order_date"])

## 7. agg_customer_lifetime — Customer LTV Metrics

In [ ]:
df_clv = (
    spark.table(f"{GOLD_SCHEMA}.fact_orders")
    .groupBy("customer_id")
    .agg(
        count("order_id").alias("total_orders"),
        spark_sum("total_amount").alias("lifetime_value"),
        avg("total_amount").alias("avg_order_value"),
        min("order_date").alias("first_order_date"),
        max("order_date").alias("last_order_date"),
        spark_sum("discount_amount").alias("total_discounts_received"),
    )
    .withColumn("lifetime_value", spark_round("lifetime_value", 2))
    .withColumn("avg_order_value", spark_round("avg_order_value", 2))
    # RFM tier classification
    .withColumn("ltv_tier",
        when(col("lifetime_value") >= 500, "Platinum")
        .when(col("lifetime_value") >= 200, "Gold")
        .when(col("lifetime_value") >= 100, "Silver")
        .otherwise("Bronze")
    )
    .withColumn("_gold_updated_at", current_timestamp())
)

write_gold(df_clv, "agg_customer_lifetime", zorder_cols=["customer_id"])

## 8. Gold Layer Summary

In [ ]:
print("=" * 60)
print("GOLD LAYER BUILD SUMMARY")
print("=" * 60)
tables = [
    "dim_date", "dim_customer", "dim_product",
    "fact_orders", "fact_order_items",
    "agg_sales_daily", "agg_customer_lifetime"
]
for table in tables:
    count = spark.table(f"{GOLD_SCHEMA}.{table}").count()
    print(f"  gold.{table:30s}: {count:>10,} rows")

# Quick KPI preview
print("\nKEY METRICS PREVIEW")
print("-" * 40)
kpis = spark.table(f"{GOLD_SCHEMA}.fact_orders").agg(
    count("order_id").alias("total_orders"),
    spark_sum("total_amount").alias("total_revenue"),
    avg("total_amount").alias("avg_order_value")
).collect()[0]
print(f"  Total Orders  : {kpis['total_orders']:,}")
print(f"  Total Revenue : ${kpis['total_revenue']:,.2f}")
print(f"  Avg Order Val : ${kpis['avg_order_value']:,.2f}")